# Code Reviewer Agent (Claude Agent SDK)

A **read-only** agent that reviews `task_project/` (`shop.py`, `test_shop.py`).

| Step | What |
|------|------|
| 1 | Data set: `task_project/` (already in this folder) |
| 2 | Implement the agent: `scout(prompt)` wraps `query(...)` |
| 3 | Run it with two prompts and print tool names, final report, turns and cost |

**Setup:** copy `.env.example` to `.env` and put your key in it. The key never goes in the notebook.

In [ ]:
%pip install claude-agent-sdk==0.2.93 python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
print("API key loaded:", os.environ.get("ANTHROPIC_API_KEY") is not None)

In [ ]:
import asyncio
import sys
import threading


def run_sync(coro_fn):
    """Run an `async def` function to completion and return its result.

    Windows: Jupyter's kernel loop is a SelectorEventLoop, which cannot spawn
    subprocesses. The Claude Agent SDK launches the Claude Code CLI as a
    subprocess, so we run on a separate thread with its own Proactor loop.
    """
    outcome = {}

    def runner():
        if sys.platform == "win32":
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        try:
            outcome["value"] = asyncio.run(coro_fn())
        except Exception as exc:
            outcome["error"] = exc

    thread = threading.Thread(target=runner)
    thread.start()
    thread.join()

    if "error" in outcome:
        raise outcome["error"]
    return outcome.get("value")

In [ ]:
MODEL_NAME = "claude-haiku-5-5"   # or "claude-sonnet-5-5" / "claude-opus-5-5"

## Step 1 — The data set

Look at the project first so we can check the agent's answers against the truth.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("task_project").resolve()
for f in sorted(PROJECT_DIR.glob("*.py")):
    print(f"--- {f.name} ---")
    print(f.read_text())

## Step 2 — The agent

* `tools=[...]` limits which built-in tools exist for the agent: only `Glob`, `Grep`, `Read`.
* `allowed_tools=[...]` lets those run without a permission prompt.
* `disallowed_tools=["Bash", ...]` is a second guard so `Bash`, `Write` and `Edit` can never run.
* `cwd=PROJECT_DIR` points the agent at the project.

`scout()` reads the stream: tool names come from `ToolUseBlock`s inside `AssistantMessage`s, and the final report, turns and cost come from the single `ResultMessage`.

In [ ]:
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    ResultMessage,
    ToolUseBlock,
)

READ_ONLY_TOOLS = ["Glob", "Grep", "Read"]


def scout(prompt):
    """Run a read-only code-review agent (Glob, Grep, Read; no Bash)."""

    async def main():
        async for message in query(
            prompt=prompt,
            options=ClaudeAgentOptions(
                tools=READ_ONLY_TOOLS,
                allowed_tools=READ_ONLY_TOOLS,
                disallowed_tools=["Bash", "Write", "Edit"],
                cwd=str(PROJECT_DIR),
                model=MODEL_NAME,
            ),
        ):
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, ToolUseBlock):
                        print(f"[tool] {block.name}")
            elif isinstance(message, ResultMessage):
                if message.is_error:
                    print("\n!! Agent run failed:", message.result or message.errors)
                print("\n=== FINAL REPORT ===")
                print(message.result)
                print("\n--- stats ---")
                print(f"turns: {message.num_turns}")
                cost = message.total_cost_usd or 0.0
                print(f"total cost: ${cost:.4f}")

    run_sync(main)

## Step 3 — Run it

### Prompt A: function inventory + missing docstrings

In [ ]:
scout("""
Review the Python project in the current directory.
List every function defined in it (file and function name),
and say which functions have no docstring.
""")

### Prompt B: bugs and security problems

In [ ]:
scout("""
Review the Python project in the current directory.
Report any bugs or security problems you find.
For each one give the file, the function, what is wrong, and a one-line fix.
""")

## Answers

**1. Did the agent find at least 3 real problems? Name them.**

Yes, if the run matches the code. `shop.py` contains these real problems, and Prompt B should surface at least three of them (check your own run's output against this list):

1. `apply_discount` calls `eval(code)` on user input, which allows arbitrary code execution. Fix: parse a number with `float()` instead.
2. `find_user` builds SQL by string concatenation, which is SQL injection. Fix: `cur.execute("... WHERE name = ?", (username,))`.
3. `ADMIN_PASSWORD = "admin123"` is a hard-coded secret. Fix: read it from an environment variable (and compare hashed values).
4. `last_item` uses `cart[len(cart)]`, which always raises `IndexError`. Fix: `cart[-1]`.
5. `average_price` divides by `len(cart)`, which raises `ZeroDivisionError` on an empty cart. Fix: return 0 when the cart is empty.
6. `new_cart(items=[])` uses a mutable default argument, so the list is shared between calls. Fix: use `items=None`.
7. `find_user` never closes the DB connection, and `average_price` ignores `qty`. Both are lower-severity.

Also, `test_cart_total` compares floats with `==` (`10 * 1.08` is not exactly `10.8`), and `test_discount` expects `90` from floating-point math, so the tests may fail or be fragile.

**2. Which message type gave you the tool names, and why is there no `ToolUseMessage`?**

The tool names came from `AssistantMessage`: its `content` list holds `ToolUseBlock`s, and `block.name` is the tool name. The tool's output comes back in a `UserMessage` as a `ToolResultBlock`.

There is no `ToolUseMessage` because the Python SDK has only four top-level message types (`SystemMessage`, `AssistantMessage`, `UserMessage`, `ResultMessage`). A tool call is part of what Claude says in one turn, so it is a **content block** inside `AssistantMessage`, next to `TextBlock`. A tool result is likewise a content block inside `UserMessage`. You find tool use with a second `isinstance()` check on each block, not on the outer message.